In [1]:
# 1. Cài đặt các thư viện lõi
!pip install filterpy lapx motmetrics cython_bbox opencv-python-headless loguru easydict scipy yacs faiss-cpu pandas matplotlib

# 2. Clone repo BoT-SORT
!git clone https://github.com/NirAharon/BoT-SORT.git

# 3. Tạo thư mục 'pretrained' BÊN TRONG BoT-SORT theo đúng tài liệu
!mkdir -p /kaggle/working/BoT-SORT/pretrained

# 4. Tải mô hình ReID (mot17_sbs_S50.pth) vào đúng thư mục vừa tạo

!gdown "1QZFWpoa80rqo7O-HXmlss8J8CnS7IUsN" -O /kaggle/working/BoT-SORT/pretrained/mot17_sbs_S50.pth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.0/178.0 kB 6.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 49.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.5/161.5 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 66.1 MB/s eta 0:00:00:00:0100:01
  Created wheel for filterpy: filename=filterpy-1.4.5-py3-none-any.whl size=110460 sha256=d6080c3235aa6947ee546525aed825bc0c9fc70c29e9d84eebf5ca3180123985
  Stored in directory: /root/.cache/pip/wheels/77/bf/4c/b0c3f4798a0166668752312a67118b27a3cd341e13ac0ae6ee
  Created wheel for cython_bbox: filename=cython_bbox-0.1.5-cp312-cp312-linux_x86_64.whl size=111563 sha256=5d854d36649cac9a0b984dcefba38b5a20f532f64a25dc8a330ad980ecbcd6b7
  Stored in directory: /root/.cache/pip/wheels/f1/e7/0a/7c310ac8921f2c1

In [2]:
import sys
import os
import glob
import cv2
import pandas as pd
import numpy as np
import torch
import collections
import collections.abc
import types
import motmetrics as mm
from tqdm.notebook import tqdm

# Nạp BoT-SORT và FastReID vào hệ thống ---
BOT_SORT_PATH = '/kaggle/working/BoT-SORT'
sys.path.insert(0, BOT_SORT_PATH) 
sys.path.insert(0, os.path.join(BOT_SORT_PATH, 'fast_reid'))

In [3]:
# --- FIX LỖI NUMPY 2.0 ---
# Định nghĩa lại hàm asfarray đã bị xóa bằng cách trỏ nó về asarray(..., dtype=float)
if not hasattr(np, 'asfarray'):
    np.asfarray = lambda x: np.asarray(x, dtype=float)
# -------------------------

# 1. Vá lỗi Python 3.10+ (ImportError: cannot import name 'Mapping'...)
collections.Mapping = collections.abc.Mapping
collections.MutableMapping = collections.abc.MutableMapping

# 2. Vá lỗi PyTorch 2.0+ (ModuleNotFoundError: No module named 'torch._six')
# Code cũ tìm 'string_classes' trong 'torch._six', ta tạo giả nó.
if 'torch._six' not in sys.modules:
    dummy_six = types.ModuleType('torch._six')
    dummy_six.string_classes = str
    sys.modules['torch._six'] = dummy_six
    # Gán ngược vào torch để chắc chắn
    if not hasattr(torch, '_six'):
        torch._six = dummy_six

# --- VÁ LỖI NUMPY (Chạy 1 lần là được) ---
# Khôi phục lại các thuộc tính đã bị xóa ở Numpy bản mới
if not hasattr(np, 'float'):
    np.float = float
if not hasattr(np, 'int'):
    np.int = int
if not hasattr(np, 'bool'):
    np.bool = bool

print("Đã vá lỗi np.float thành công!")

Đã vá lỗi np.float thành công!


In [4]:
# TARGET_CLASSES = ['Car', 'Pedestrian', 'Cyclist']

# def parse_kitti_tracking_file(label_path):
#     if not os.path.exists(label_path):
#         print(f"File not found: {label_path}")
#         return {}, pd.DataFrame()
    
#     columns = [
#         'frame', 'track_id', 'type', 'truncated', 'occluded', 'alpha', 
#         'bbox_l', 'bbox_t', 'bbox_r', 'bbox_b', 
#         'dim_h', 'dim_w', 'dim_l', 'loc_x', 'loc_y', 'loc_z', 'rot_y'
#     ]
#     df = pd.read_csv(label_path, sep=' ', names=columns)
#     df = df[df['track_id'] != -1]
#     #df = df[df['type'] == 'Car']
#     df = df[df['type'] == target_class]
    
#     detections = {}
#     for frame_idx, group in df.groupby('frame'):
#         dets_list = []
#         for _, row in group.iterrows():
#             det = {
#                 'boxs': [row['bbox_l'], row['bbox_t'], row['bbox_r'], row['bbox_b']],
#                 'score': 0.99, 
#             }
#             dets_list.append(det)
#         detections[frame_idx] = dets_list
        
#     return detections, df

In [5]:
from tracker.bot_sort import BoTSORT

class BoTSORTArgs:
    def __init__(self):
        self.track_high_thresh = 0.5
        self.track_low_thresh = 0.2
        self.new_track_thresh = 0.7
        self.match_thresh = 0.8
        self.track_buffer = 30
        self.mot20 = False
        self.cmc_method = "sparseOptFlow" 
        self.name = "bot_sort"
        self.ablation = False 
        self.fps = 30

        self.with_reid = True
        # File config của mạng SBS-S50 có sẵn trong thư mục clone
        self.fast_reid_config = r"/kaggle/working/BoT-SORT/fast_reid/configs/MOT17/sbs_S50.yml"
        # Trọng số vừa tải vào thư mục pretrained nội bộ
        self.fast_reid_weights = r"/kaggle/working/BoT-SORT/pretrained/mot17_sbs_S50.pth"
        self.proximity_thresh = 0.5
        self.appearance_thresh = 0.25
        self.aspect_ratio_thresh = 100.0
        self.device = "cuda" if torch.cuda.is_available() else "cpu"

2026-03-14 17:16:58.965486: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773508619.159660      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773508619.214005      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773508619.633758      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773508619.633799      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773508619.633802      55 computation_placer.cc:177] computation placer alr

In [6]:
TARGET_CLASSES = ['Car', 'Pedestrian', 'Cyclist']

def parse_kitti_tracking_file(label_path):
    if not os.path.exists(label_path):
        print(f"File not found: {label_path}")
        return {}, pd.DataFrame()
    
    columns = [
        'frame', 'track_id', 'type', 'truncated', 'occluded', 'alpha', 
        'bbox_l', 'bbox_t', 'bbox_r', 'bbox_b', 
        'dim_h', 'dim_w', 'dim_l', 'loc_x', 'loc_y', 'loc_z', 'rot_y'
    ]
    df = pd.read_csv(label_path, sep=' ', names=columns)
    df = df[df['track_id'] != -1]
    df = df[df['type'].isin(TARGET_CLASSES)]
    
    detections = {}
    for frame_idx, group in df.groupby('frame'):
        frame_dets = {c: [] for c in TARGET_CLASSES}
        for _, row in group.iterrows():
            obj_type = row['type']
            det = {
                'boxs': [row['bbox_l'], row['bbox_t'], row['bbox_r'], row['bbox_b']],
                'score': 0.99, 
            }
            frame_dets[obj_type].append(det)
        detections[frame_idx] = frame_dets
        
    return detections, df

# Scale lên toàn bộ data

In [7]:
import time

KAGGLE_ROOT = "/kaggle/input/datasets/leducnhuan/kitti-tracking/kitti_tracking/training/image_02"
LABEL_ROOT = "/kaggle/input/datasets/leducnhuan/kitti-tracking/kitti_tracking/training/label_02"
# Lấy danh sách sequence (0000,0001,...)
SEQ_LIST = sorted(os.listdir(KAGGLE_ROOT))
print("Sequences to process:", SEQ_LIST)

# Khởi tạo thông số BoTSORT chung
args = BoTSORTArgs()
args.min_box_area = 10        # Tắt màng lọc hộp nhỏ
args.mot20 = False            # Tắt luật rườm rà của MOT20
mh = mm.metrics.create()

all_accs = {c: [] for c in TARGET_CLASSES}
all_names = {c: [] for c in TARGET_CLASSES}

tracking_time = 0
tracking_calls = 0

for SEQ_ID in SEQ_LIST:
    print(f"\n========== Processing Sequence {SEQ_ID} ==========")
    img_dir = os.path.join(KAGGLE_ROOT, SEQ_ID)
    label_file = os.path.join(LABEL_ROOT, f"{SEQ_ID}.txt")
    if not os.path.exists(label_file):
        print(f" Skip {SEQ_ID}, no label file")
        continue
        
    # Load detections & GT
    seq_detections, gt_df = parse_kitti_tracking_file(label_file)
    image_paths = sorted(glob.glob(os.path.join(img_dir, "*.png")))
    print(f"Frames: {len(image_paths)}")
    
    # =========================
    # RESET TRACKER & ACCUMULATOR PER SEQ
    # =========================
    # KHỞI TẠO MỚI CHO MỖI VIDEO
    
    trackers = {c: BoTSORT(args, frame_rate=30) for c in TARGET_CLASSES}
    accs = {c: mm.MOTAccumulator(auto_id=True) for c in TARGET_CLASSES}
    
    # =========================
    # FRAME LOOP
    # =========================

    for k, img_path in enumerate(tqdm(image_paths, leave=False)):
        img = cv2.imread(img_path)
        if img is None:
            continue
        img_h, img_w = img.shape[0], img.shape[1]
        img_info = [img_h, img_w]
        
        # 1. Format Detections (N, 5)
        frame_dets_all_classes = seq_detections.get(k, {c: [] for c in TARGET_CLASSES})
        for class_name in TARGET_CLASSES:
            dets_list = []
            for det in frame_dets_all_classes[class_name]:
                box = det['boxs']
                score = det['score']
                x1 = max(0, min(box[0], img_w - 1))
                y1 = max(0, min(box[1], img_h - 1))
                x2 = max(0, min(box[2], img_w - 1))
                y2 = max(0, min(box[3], img_h - 1))  
                w = x2 - x1
                h = y2 - y1  
                if w <= 0 or h <= 0:
                    continue  
                dets_list.append([x1, y1, x2, y2, score])
                
            dets_array = np.array(dets_list, dtype=np.float32) if len(dets_list) > 0 else np.empty((0, 5), dtype=np.float32)
            # 2. Update BoTSORT
            try:
                start = time.time()
                online_targets = trackers[class_name].update(dets_array,img)
                tracking_time += time.time() - start
                tracking_calls += 1
            except Exception:
                print(f"Tracker error at seq {SEQ_ID}, frame {k}, class {class_name}")
                continue
                
            # 3. Collect Hypothesis (BoTSORT Output)
            frame_hypos = []
            frame_hypo_ids = []
            for t in online_targets:
                tlwh = t.tlwh 
                frame_hypos.append([tlwh[0], tlwh[1], tlwh[2], tlwh[3]])
                frame_hypo_ids.append(t.track_id)
        
            # 4. Collect Ground Truth
            frame_gt = []
            frame_gt_ids = []
    
            if not gt_df.empty:
                gt_now = gt_df[(gt_df['frame'] == k) & (gt_df['type'] == class_name)]
                for _, row in gt_now.iterrows():
                    if int(row['track_id']) == -1:
                        continue
                    w = row['bbox_r'] - row['bbox_l']
                    h = row['bbox_b'] - row['bbox_t']
                    frame_gt.append([row['bbox_l'], row['bbox_t'], w, h])
                    frame_gt_ids.append(int(row['track_id']))

            # 5. Update Metrics
            dist_matrix = mm.distances.iou_matrix(frame_gt, frame_hypos, max_iou=0.5)
            accs[class_name].update(frame_gt_ids, frame_hypo_ids, dist_matrix)
            
    # Lưu lại bộ thu thập metrics của sequence này
    for c in TARGET_CLASSES:
        all_accs[c].append(accs[c])
        all_names[c].append(SEQ_ID)

print("\nTracking Completed.")
print("\n========== TRACKING SPEED ==========")
print(f"Total tracker calls: {tracking_calls}")
print(f"Total tracking time: {tracking_time:.3f} seconds")
print(f"Average time per call: {(tracking_time/tracking_calls)*1000:.3f} ms")

Sequences to process: ['0000', '0001', '0002', '0003', '0004', '0005', '0006', '0007', '0008', '0009', '0010', '0011', '0012', '0013', '0014', '0015', '0016', '0017', '0018', '0019', '0020']

========== Processing Sequence 0000 ==========
Frames: 154


Skip loading parameter 'heads.weight' to the model due to incompatible shapes: (487, 2048) in the checkpoint but (0, 2048) in the model! You might want to double check if this is expected.
Skip loading parameter 'heads.weight' to the model due to incompatible shapes: (487, 2048) in the checkpoint but (0, 2048) in the model! You might want to double check if this is expected.
Skip loading parameter 'heads.weight' to the model due to incompatible shapes: (487, 2048) in the checkpoint but (0, 2048) in the model! You might want to double check if this is expected.


  0%|          | 0/154 [00:00<?, ?it/s]


========== Processing Sequence 0001 ==========
Frames: 447


Skip loading parameter 'heads.weight' to the model due to incompatible shapes: (487, 2048) in the checkpoint but (0, 2048) in the model! You might want to double check if this is expected.
Skip loading parameter 'heads.weight' to the model due to incompatible shapes: (487, 2048) in the checkpoint but (0, 2048) in the model! You might want to double check if this is expected.
Skip loading parameter 'heads.weight' to the model due to incompatible shapes: (487, 2048) in the checkpoint but (0, 2048) in the model! You might want to double check if this is expected.


...

  0%|          | 0/1059 [00:00<?, ?it/s]


========== Processing Sequence 0020 ==========
Frames: 837


Skip loading parameter 'heads.weight' to the model due to incompatible shapes: (487, 2048) in the checkpoint but (0, 2048) in the model! You might want to double check if this is expected.
Skip loading parameter 'heads.weight' to the model due to incompatible shapes: (487, 2048) in the checkpoint but (0, 2048) in the model! You might want to double check if this is expected.
Skip loading parameter 'heads.weight' to the model due to incompatible shapes: (487, 2048) in the checkpoint but (0, 2048) in the model! You might want to double check if this is expected.


  0%|          | 0/837 [00:00<?, ?it/s]


Tracking Completed.

========== TRACKING SPEED ==========
Total tracker calls: 24024
Total tracking time: 803.833 seconds
Average time per call: 33.460 ms


In [9]:
all_class_summaries = []

for class_name in TARGET_CLASSES:
    summary_class = mh.compute_many(
        all_accs[class_name], 
        metrics=['num_frames', 'mota', 'motp', 'idf1', 'idp', 'idr', 'num_switches'], 
        names=all_names[class_name],
        generate_overall=True
    )

    # Quan trọng: bỏ các sequence có MOTP NaN trước khi aggregate
    if 'OVERALL' in summary_class.index:
        motp_vals = summary_class.loc[summary_class.index != 'OVERALL', 'motp']
        summary_class.loc['OVERALL', 'motp'] = motp_vals.mean(skipna=True)

    summary_class.insert(0, 'Class', class_name)
    all_class_summaries.append(summary_class)


# Ghép bảng
final_df = pd.concat(all_class_summaries)
final_df.reset_index(inplace=True)
final_df.rename(columns={'index': 'Sequence'}, inplace=True)

final_df = final_df.sort_values(by=['Sequence', 'Class'])
final_df.set_index(['Sequence', 'Class'], inplace=True)

display(final_df)

num_frames      mota      motp      idf1       idp  \
Sequence Class                                                            
0000     Car                154  1.000000  0.036476  1.000000  1.000000   
         Cyclist            154  1.000000  0.056186  1.000000  1.000000   
         Pedestrian         154  1.000000  0.113015  1.000000  1.000000   
0001     Car                447  0.995897  0.058670  0.991048  0.991048   
         Cyclist            447       NaN       NaN       NaN       NaN   
...                         ...       ...       ...       ...       ...   
0020     Cyclist            837       NaN       NaN       NaN       NaN   
         Pedestrian         837       NaN       NaN       NaN       NaN   
OVERALL  Car               8008  0.995714  0.045925  0.972454  0.972454   
         Cyclist           8008  0.992260  0.065153  0.975748  0.975748   
         Pedestrian        8008  0.988928  0.084069  0.952788  0.952829   

                          idr  num_switches  
Sequence Class                               
0000     Car         1.000000             0  
         Cyclist     1.000000             0  
         Pedestrian  1.000000             0  
0001     Car         0.991048            11  
         Cyclist          NaN             0  
...                       ...           ...  
0020     Cyclist          NaN             0  
         Pedestrian       NaN             0  
OVERALL  Car         0.972454           105  
         Cyclist     0.975748            13  
         Pedestrian  0.952746           118  

[66 rows x 7 columns]

In [11]:
save_path = "/kaggle/working/botsort_ReID_kitti_results.csv"
final_df.to_csv(save_path, float_format='%.3f')
print("Saved to:", save_path)

Saved to: /kaggle/working/botsort_ReID_kitti_results.csv
